# BottleneckOracle — Timing-Based DAG Construction

**Fix**: Edges are built from real PyTorch Profiler timing data.  
Two ops get an edge only when one finishes **before** the other starts (`end_time <= ts`).  
Overlapping (parallel) ops have **no** edge, producing genuine variance in `is_critical` labels.

**Success condition (Cell 9):** `label_std > 0` and `0.05 < critical_fraction < 0.5`

In [1]:
# Cell 1 — Installs
!pip install -q torch torchvision torch-geometric huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.3 MB/s eta 0:00:00


In [2]:
# Cell 2 — Imports
import torch
import torch.nn as nn
from torch.profiler import ProfilerActivity, profile, record_function
from torch_geometric.data import HeteroData
import networkx as nx
import json
import os
import numpy as np
from scipy.stats import spearmanr

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

PyTorch version : 2.10.0+cu128
CUDA available  : True


In [3]:
# Cell 3 — Model configs
configs = [
    {"name": "tiny",   "d_model": 64,  "nhead": 2, "num_layers": 1},
    {"name": "small",  "d_model": 128, "nhead": 4, "num_layers": 2},
    {"name": "medium", "d_model": 256, "nhead": 8, "num_layers": 4},
]

os.makedirs("data/raw",    exist_ok=True)
os.makedirs("data/graphs", exist_ok=True)

print("Configs defined:", [c["name"] for c in configs])

Configs defined: ['tiny', 'small', 'medium']


In [4]:
# Cell 4 — generate_trace(config, trace_id)
def generate_trace(config: dict, trace_id: int) -> str:
    """Profile a nn.Transformer and export a chrome-trace JSON.

    Returns the path to the JSON file.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = nn.Transformer(
        d_model=config["d_model"],
        nhead=config["nhead"],
        num_encoder_layers=config["num_layers"],
        num_decoder_layers=config["num_layers"],
        batch_first=True,
    ).to(device)

    seq_len, batch = 32, 4
    src = torch.randn(batch, seq_len, config["d_model"]).to(device)
    tgt = torch.randn(batch, seq_len, config["d_model"]).to(device)

    trace_path = f"data/raw/trace_{config['name']}_{trace_id}.json"

    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        with_stack=False,
    ) as prof:
        with record_function("model_inference"):
            with torch.no_grad():
                _ = model(src, tgt)

    prof.export_chrome_trace(trace_path)
    return trace_path

print("generate_trace defined.")

generate_trace defined.


In [5]:
def parse_trace(filepath: str) -> nx.DiGraph:
    """Parse a chrome-trace JSON into a NetworkX DiGraph.

    Edge rule (TIMING-BASED with JOIN POINTS):
        For each op_i, find ALL ops_j (j > i by timestamp) such that
        op_i.end_time <= op_j.ts  (op_i fully finishes before op_j starts).
        Add edge i → j for ALL such ops (not just the first one).

    This creates proper join points where network nodes can depend on
    multiple compute nodes (e.g., allreduce after several matmuls).
    """
    with open(filepath, "r") as f:
        trace = json.load(f)

    events = trace.get("traceEvents", trace) if isinstance(trace, dict) else trace

    # ── 1. Filter: keep cpu_op events that have a duration ──────────────────
    ops = [
        e for e in events
        if e.get("cat") == "cpu_op" and "dur" in e and "ts" in e
    ]

    if not ops:
        raise ValueError(f"No cpu_op events with 'dur' found in {filepath}")

    # ── 2. Sort by start timestamp ───────────────────────────────────────────
    ops.sort(key=lambda e: e["ts"])

    # ── 3. Build the DiGraph ─────────────────────────────────────────────────
    G = nx.DiGraph()

    for idx, op in enumerate(ops):
        ts       = op["ts"]
        dur      = op["dur"]
        end_time = ts + dur
        name     = op.get("name", f"op_{idx}")
        op_type  = (
            "network"
            if any(kw in name.lower() for kw in ("allreduce", "comm", "nccl", "send", "recv"))
            else "compute"
        )

        G.add_node(
            idx,
            name=name,
            type=op_type,
            duration_ms=dur / 1000.0,
            timestamp=ts,
            end_time=end_time,
        )

    # ── 4. TIMING-BASED edges with JOIN POINTS ────────────────────────────────
    #   For each op_i, connect to ALL ops_j that start after op_i ends.
    #   This creates proper join points where multiple computes can feed into
    #   one network node (e.g., allreduce waiting for multiple matmuls).
    for i in range(len(ops)):
        end_i = G.nodes[i]["end_time"]
        for j in range(i + 1, len(ops)):
            ts_j = G.nodes[j]["timestamp"]
            if end_i <= ts_j:          # op_i fully before op_j → real dependency
                G.add_edge(i, j)
            # NO BREAK - connect to ALL non-overlapping successors, not just first

    print(f"  Parsed {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # ── 5. Validate join points ────────────────────────────────────────────────
    join_nodes = [n for n in G.nodes() if G.in_degree(n) >= 2]
    network_joins = [n for n in join_nodes if G.nodes[n]["type"] == "network"]

    if network_joins:
        print(f"  Found {len(network_joins)} network join points (in-degree ≥ 2)")
        # Verify timing consistency for join nodes
        eps = 1e-6  # Small epsilon for floating point comparison
        for node in network_joins[:3]:  # Check first 3 as samples
            node_start = G.nodes[node]["timestamp"]
            pred_ends = [G.nodes[p]["end_time"] for p in G.predecessors(node)]
            max_pred_end = max(pred_ends)
            if abs(node_start - max_pred_end) > eps:
                print(f"    ⚠ WARNING: Node {node} starts at {node_start:.3f}, "
                      f"but max predecessor end is {max_pred_end:.3f}")

    return G

print("parse_trace defined with JOIN POINT support.")

parse_trace defined with JOIN POINT support.


In [6]:
# Cell 6 — find_critical_path(G)
def find_critical_path(G: nx.DiGraph):
    """Return (critical_path_nodes, critical_path_length_ms).

    Edge weight is set to the source node's duration_ms so that
    the longest path = the path that accumulates the most compute time.
    """
    if G.number_of_edges() == 0:
        # Degenerate graph — all nodes are isolated, pick the heaviest
        heaviest = max(G.nodes, key=lambda n: G.nodes[n]["duration_ms"])
        return [heaviest], G.nodes[heaviest]["duration_ms"]

    # Assign edge weights equal to the *source* node duration
    for u, v in G.edges():
        G[u][v]["weight"] = G.nodes[u]["duration_ms"]

    # Ensure the graph is a DAG (remove any cycles introduced by floating-point ties)
    if not nx.is_directed_acyclic_graph(G):
        G = nx.DiGraph(nx.dag.transitive_reduction(G))  # fallback

    critical_path   = nx.dag_longest_path(G, weight="weight")
    critical_length = nx.dag_longest_path_length(G, weight="weight")

    return critical_path, critical_length

print("find_critical_path defined.")

find_critical_path defined.


In [7]:
def add_features(G: nx.DiGraph, critical_path: list) -> nx.DiGraph:
    """Annotate each node with compute_ratio, is_critical, norm_duration, and CPM slack.

    CPM (Critical Path Method) computation:
    - Forward pass: Compute ES (earliest start) and EF (earliest finish)
    - Backward pass: Compute LS (latest start) and LF (latest finish)
    - Slack = LS - ES = LF - EF (how much delay each operation can absorb)

    Prints the fraction of critical nodes — should be well below 1.0.
    """
    critical_set   = set(critical_path)
    total_time     = sum(G.nodes[n]["duration_ms"] for n in G.nodes)
    max_duration   = max(G.nodes[n]["duration_ms"] for n in G.nodes)

    # ── CPM: Forward pass (ES/EF) ────────────────────────────────────────────
    # ES(node) = max(EF of all predecessors) or 0 if no predecessors
    # EF(node) = ES(node) + duration_ms
    for n in nx.topological_sort(G):
        predecessors = list(G.predecessors(n))
        if predecessors:
            es = max(G.nodes[p]["ef"] for p in predecessors)
        else:
            es = 0.0
        ef = es + G.nodes[n]["duration_ms"]
        G.nodes[n]["es"] = es
        G.nodes[n]["ef"] = ef

    # ── CPM: Backward pass (LS/LF) ────────────────────────────────────────────
    # LF(node) = min(LS of all successors) or project_end_time if no successors
    # LS(node) = LF(node) - duration_ms
    # Start from nodes with no successors (sinks) and work backwards
    project_end_time = max(G.nodes[n]["ef"] for n in G.nodes)

    # Reverse topological order for backward pass
    for n in reversed(list(nx.topological_sort(G))):
        successors = list(G.successors(n))
        if successors:
            lf = min(G.nodes[s]["ls"] for s in successors)
        else:
            lf = project_end_time
        ls = lf - G.nodes[n]["duration_ms"]
        G.nodes[n]["ls"] = ls
        G.nodes[n]["lf"] = lf

    # ── Compute Slack ──────────────────────────────────────────────────────────
    # Slack = LS - ES (or LF - EF)
    # Slack = 0 means node is on critical path
    # Slack > 0 means node can be delayed without affecting project completion
    for n in G.nodes:
        dur = G.nodes[n]["duration_ms"]
        slack = G.nodes[n]["ls"] - G.nodes[n]["es"]  # Should equal LF - EF
        G.nodes[n]["slack"] = max(0.0, slack)  # Ensure non-negative (handles floating point errors)
        G.nodes[n]["compute_ratio"]  = dur / total_time if total_time > 0 else 0.0
        G.nodes[n]["is_critical"]    = 1 if n in critical_set else 0  # Keep for backward compatibility
        G.nodes[n]["norm_duration"]  = dur / max_duration if max_duration > 0 else 0.0

    # ── Statistics ───────────────────────────────────────────────────────────────
    n_critical = sum(1 for n in G.nodes if G.nodes[n]["is_critical"] == 1)
    fraction   = n_critical / G.number_of_nodes()

    # Slack statistics
    slack_values = [G.nodes[n]["slack"] for n in G.nodes]
    avg_slack = sum(slack_values) / len(slack_values)
    max_slack = max(slack_values)
    zero_slack_count = sum(1 for s in slack_values if abs(s) < 1e-6)

    print(f"  Critical nodes: {n_critical}/{G.number_of_nodes()} "
          f"({fraction:.2%}) — {'✓ GOOD' if 0.05 < fraction < 0.5 else '✗ CHECK EDGE LOGIC'}")
    print(f"  Slack statistics: avg={avg_slack:.3f}ms, max={max_slack:.3f}ms, "
          f"zero_slack={zero_slack_count} nodes")

    return G

print("add_features defined with CPM slack computation.")

add_features defined with CPM slack computation.


In [8]:
def to_pyg(G: nx.DiGraph, total_time: float) -> HeteroData:
    """Convert annotated DiGraph to a PyG HeteroData object.

    Node features: [duration_ms, compute_ratio, norm_duration, slack_normalized]
    Node labels  : slack (continuous target for regression) AND is_critical (binary for reference)
    Graph label  : total_time (ms)

    IMPORTANT: Changed from binary criticality to continuous slack as the main prediction target.
    This better captures the spectrum of node importance and improves OOD generalization.
    """
    data = HeteroData()

    # ── Separate compute vs. network nodes ──────────────────────────────────
    compute_nodes = [n for n in G.nodes if G.nodes[n]["type"] == "compute"]
    network_nodes = [n for n in G.nodes if G.nodes[n]["type"] == "network"]

    # ── Normalize slack for numerical stability ───────────────────────────────
    slack_values = [G.nodes[n]["slack"] for n in compute_nodes]
    max_slack = max(slack_values) if slack_values else 1.0
    slack_norm_factor = max_slack if max_slack > 0 else 1.0

    # ── compute node features & labels ──────────────────────────────────────
    if compute_nodes:
        compute_feats = [
            [
                G.nodes[n]["duration_ms"],
                G.nodes[n]["compute_ratio"],
                G.nodes[n]["norm_duration"],
                G.nodes[n]["slack"] / slack_norm_factor,  # Normalized slack as feature
            ]
            for n in compute_nodes
        ]
        data["compute"].x = torch.tensor(compute_feats, dtype=torch.float)

        # MAIN LABEL: Continuous slack for regression
        data["compute"].y = torch.tensor(
            [G.nodes[n]["slack"] for n in compute_nodes],
            dtype=torch.float,
        )

        # AUXILIARY LABEL: Binary criticality for reference/backward compatibility
        data["compute"].y_binary = torch.tensor(
            [G.nodes[n]["is_critical"] for n in compute_nodes],
            dtype=torch.float,
        )
    else:
        data["compute"].x = torch.zeros((0, 4), dtype=torch.float)
        data["compute"].y = torch.zeros(0, dtype=torch.float)
        data["compute"].y_binary = torch.zeros(0, dtype=torch.float)

    # ── network node features ────────────────────────────────────────────────
    if network_nodes:
        network_feats = [
            [
                G.nodes[n]["duration_ms"],
                G.nodes[n]["compute_ratio"],
                G.nodes[n]["norm_duration"],
                G.nodes[n]["slack"] / slack_norm_factor,  # Normalized slack as feature
            ]
            for n in network_nodes
        ]
        data["network"].x = torch.tensor(network_feats, dtype=torch.float)

    # ── compute → compute edges ──────────────────────────────────────────────
    compute_set  = set(compute_nodes)
    node_to_idx  = {n: i for i, n in enumerate(compute_nodes)}

    cc_edges = [
        (node_to_idx[u], node_to_idx[v])
        for u, v in G.edges()
        if u in compute_set and v in compute_set
    ]

    if cc_edges:
        src, dst = zip(*cc_edges)
        data["compute", "depends_on", "compute"].edge_index = torch.tensor(
            [list(src), list(dst)], dtype=torch.long
        )
    else:
        data["compute", "depends_on", "compute"].edge_index = torch.zeros(
            (2, 0), dtype=torch.long
        )

    # ── graph-level label ────────────────────────────────────────────────────
    data.y = torch.tensor([total_time], dtype=torch.float)

    return data

print("to_pyg defined with continuous slack labels.")

to_pyg defined with continuous slack labels.


In [9]:
# Cell 9 — Sanity check on one trace
print("═" * 55)
print("SANITY CHECK — single trace (tiny config, trace_id=0)")
print("═" * 55)

# 1. Generate
trace_path = generate_trace(configs[0], trace_id=0)
print(f"Trace written to: {trace_path}")

# 2. Parse
G = parse_trace(trace_path)

# 3. Critical path
cp, cp_len = find_critical_path(G)
print(f"  Critical path length : {cp_len:.3f} ms  ({len(cp)} hops)")

# 4. Features
total_time = sum(G.nodes[n]["duration_ms"] for n in G.nodes)
G = add_features(G, cp)

# 5. PyG conversion
pyg = to_pyg(G, total_time)

slack_labels = pyg["compute"].y.numpy()
binary_labels = pyg["compute"].y_binary.numpy()
slack_std  = float(slack_labels.std())
slack_mean = float(slack_labels.mean())
n_nodes    = pyg["compute"].x.shape[0]
n_edges    = pyg["compute", "depends_on", "compute"].edge_index.shape[1]
binary_crit_frac = float(binary_labels.mean())

print("\n── PyG object stats ──────────────────────")
print(f"  Compute nodes  : {n_nodes}")
print(f"  Compute edges  : {n_edges}")
print(f"  Binary critical frac  : {binary_crit_frac:.3f}")
print(f"  Slack mean     : {slack_mean:.3f} ms")
print(f"  Slack std      : {slack_std:.3f} ms")
print()

# Updated success criteria for slack-based labels
if slack_std == 0:
    print("✗ FAIL — slack_std is 0. All nodes have identical slack values.")
    print("  Check the CPM computation logic in add_features().")
elif not (0.05 < binary_crit_frac < 0.5):
    print(f"⚠ WARNING — binary critical fraction {binary_crit_frac:.3f} is outside [0.05, 0.5].")
    print("  Consider relaxing/tightening the timing threshold.")
elif slack_mean < 0:
    print("✗ FAIL — negative slack values detected. Check CPM logic.")
else:
    print("✓ PASS — slack_std > 0, binary critical fraction looks healthy, and all slacks are non-negative.")

═══════════════════════════════════════════════════════
SANITY CHECK — single trace (tiny config, trace_id=0)
═══════════════════════════════════════════════════════


/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


Trace written to: data/raw/trace_tiny_0.json
  Parsed 382 nodes, 72230 edges
  Critical path length : 422.623 ms  (110 hops)
  Critical nodes: 110/382 (28.80%) — ✓ GOOD
  Slack statistics: avg=1.556ms, max=43.704ms, zero_slack=106 nodes

── PyG object stats ──────────────────────
  Compute nodes  : 382
  Compute edges  : 72230
  Binary critical frac  : 0.288
  Slack mean     : 1.556 ms
  Slack std      : 6.157 ms

✓ PASS — slack_std > 0, binary critical fraction looks healthy, and all slacks are non-negative.


In [10]:
# Cell 10 — Generate 501 graphs (167 per config)
TRACES_PER_CONFIG = 167
total_generated   = 0
skipped           = 0

print(f"Generating {len(configs) * TRACES_PER_CONFIG} graphs total …\n")

for cfg in configs:
    print(f"── Config: {cfg['name']} ──────────────────────────────")
    for trace_id in range(TRACES_PER_CONFIG):
        save_path = f"data/graphs/graph_{cfg['name']}_{trace_id}.pt"
        if os.path.exists(save_path):
            skipped += 1
            continue

        try:
            tp   = generate_trace(cfg, trace_id)
            G    = parse_trace(tp)
            cp, cp_len = find_critical_path(G)
            total_t    = sum(G.nodes[n]["duration_ms"] for n in G.nodes)
            G    = add_features(G, cp)
            pyg  = to_pyg(G, total_t)
            torch.save(pyg, save_path)
            total_generated += 1

            if (total_generated + skipped) % 50 == 0:
                print(
                    f"  Progress: {total_generated + skipped} / "
                    f"{len(configs) * TRACES_PER_CONFIG} "
                    f"({skipped} skipped)"
                )

        except Exception as exc:
            print(f"  [SKIP] trace_id={trace_id} — {exc}")

print(f"\nDone. Generated {total_generated} new graphs, skipped {skipped} existing.")

Generating 501 graphs total …

── Config: tiny ──────────────────────────────
  Parsed 382 nodes, 72230 edges
  Critical nodes: 110/382 (28.80%) — ✓ GOOD
  Slack statistics: avg=0.017ms, max=0.114ms, zero_slack=106 nodes
  Parsed 382 nodes, 72230 edges
  Critical nodes: 110/382 (28.80%) — ✓ GOOD
  Slack statistics: avg=0.014ms, max=0.073ms, zero_slack=106 nodes
  Parsed 382 nodes, 72230 edges
  Critical nodes: 110/382 (28.80%) — ✓ GOOD
  Slack statistics: avg=0.022ms, max=0.103ms, zero_slack=106 nodes
  Parsed 382 nodes, 72230 edges
  Critical nodes: 110/382 (28.80%) — ✓ GOOD
  Slack statistics: avg=0.022ms, max=0.115ms, zero_slack=106 nodes
  Parsed 382 nodes, 72230 edges
  Critical nodes: 110/382 (28.80%) — ✓ GOOD
  Slack statistics: avg=0.025ms, max=0.122ms, zero_slack=106 nodes
  Parsed 382 nodes, 72230 edges
  Critical nodes: 110/382 (28.80%) — ✓ GOOD
  Slack statistics: avg=0.023ms, max=0.121ms, zero_slack=106 nodes
  Parsed 382 nodes, 72230 edges
  Critical nodes: 110/382 (28.80

In [12]:
import os
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Zip the local data folder (much faster than copying individual files)
!zip -r /content/data_backup.zip /content/data

# 3. Create a destination folder in your Drive
!mkdir -p "/content/drive/MyDrive/BottleneckOracle_Backup"

# 4. Copy the zipped file to your Drive
!cp /content/data_backup.zip "/content/drive/MyDrive/BottleneckOracle_Backup/"

print("Backup complete! You can safely disconnect the runtime.")

Mounted at /content/drive
  adding: content/data/ (stored 0%)
  adding: content/data/raw/ (stored 0%)
  adding: content/data/raw/trace_small_66.json (deflated 92%)
  adding: content/data/raw/trace_small_165.json (deflated 92%)
  adding: content/data/raw/trace_tiny_95.json (deflated 92%)
  adding: content/data/raw/trace_small_63.json (deflated 92%)
  adding: content/data/raw/trace_tiny_162.json (deflated 92%)
  adding: content/data/raw/trace_tiny_163.json (deflated 92%)
  adding: content/data/raw/trace_tiny_70.json (deflated 92%)
  adding: content/data/raw/trace_tiny_18.json (deflated 92%)
  adding: content/data/raw/trace_small_42.json (deflated 92%)
  adding: content/data/raw/trace_medium_150.json (deflated 93%)
  adding: content/data/raw/trace_medium_44.json (deflated 93%)
  adding: content/data/raw/trace_small_75.json (deflated 92%)
  adding: content/data/raw/trace_tiny_126.json (deflated 92%)
  adding: content/data/raw/trace_tiny_149.json (deflated 92%)
  adding: content/data/raw/tr

In [11]:
from huggingface_hub import login, HfApi
import os

# Step 1 - login with your token directly
TOKEN = ""  # paste your token here
login(token=TOKEN)

# Step 2 - verify login worked
api = HfApi()
user = api.whoami()
print(f"Logged in as: {user['name']}")

# Step 3 - upload
api.upload_folder(
    folder_path="data/graphs",
    repo_id="preethamvj/bottleneck-oracle-graphs",
    repo_type="dataset",
    delete_patterns=["*"],
    token=TOKEN  # pass token explicitly
)

print("Upload complete!")

LocalProtocolError: Illegal header value b'Bearer '

In [ ]:
from huggingface_hub import snapshot_download
import torch

path = snapshot_download(repo_id="preethamvj/bottleneck-oracle-graphs", repo_type="dataset")

for name in ["graph_tiny_0", "graph_small_0", "graph_medium_0"]:
    g = torch.load(f"{path}/{name}.pt", weights_only=False)
    labels = g['compute'].y
    print(f"{name}: nodes={labels.shape[0]}, critical={labels.sum().item():.0f}, std={labels.std().item():.4f}")